# Colab — Variant Os refit ("O simple")

Trains the regime LSTMs and computes the ensemble for **variant Os** — a new pipeline that uses variant O's price/volume features but pairs them with a plain `GaussianHMM` instead of the BIC-optimal `GMMHMM(n_mix=2)` used by the original variant O.

**Why this exists.** The BIC-optimal `GMMHMM(n_mix=2)` on variant O's price/volume features converges to a non-regime basin (lag-1 autocorrelation of $p_{\mathrm{volatile}} \approx 0.17$, ~2,300 flips above/below 0.5 across the test period, median run length 2 days). This violates volatility-clustering as a stylized fact and makes the variant-O regime label essentially noise. Replacing GMMHMM with a plain `GaussianHMM` on the same features produces clean regime structure (autocorr 0.98, 97 flips, persistent runs). The original variant O is preserved unchanged for direct comparison; this notebook trains the LSTMs for the *new* `Os` variant. Variants A and B are unaffected — their richer features anchor GMMHMM into a regime basin.

**Evidence and diagnostics:** `supplementary/diagnostics/variant_O_hmm_sweep.csv`, `supplementary/figures/F_variant_O_hmm_diagnostic.pdf`, `supplementary/figures/F4_p_volatile_over_time.pdf`.

**Self-contained:** this notebook does NOT modify `nb05` or `nb06`. It uses `src/train_LSTM_regime.py` directly (the same script `nb05` uses) and computes the ensemble + DM test inline. All output filenames use the `_Os` suffix so they are non-overlapping with existing variant O / A / H / B artifacts.

**Wall time:** ~25–35 min on Colab GPU (3 seeds × 2 regimes = 6 LSTM trainings; seed 42 runs the full 20-trial Optuna search, seeds 43/44 reuse seed-42 hparams).

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DRIVE = '/content/drive/MyDrive/StockVolatilitySight'
os.makedirs(PROJECT_DRIVE, exist_ok=True)
print('Drive root:', PROJECT_DRIVE)

## Cell 2 — Clone repo (uses your classic GitHub PAT)

Set `GH_PAT` below. The token only needs `repo` scope.

Make sure the branch you clone has the `models/hmm_*_Os.joblib` and `data/processed/regime_probabilities_Os.parquet` files committed (these are produced by the local refit script `tmp/save_sweep_and_train_O.py` — verify locally before committing/pushing).

In [ ]:
GH_PAT = 'PASTE_YOUR_TOKEN_HERE'   # classic PAT, repo scope
GH_USER = 'maharajhaider'
GH_REPO = 'StockVolatilitySight'
GH_BRANCH = 'iteration-and-report-analysis'   # branch with the variant-Os fix

import subprocess, shutil, os
REPO_DIR = '/content/repo'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
url = f'https://{GH_PAT}@github.com/{GH_USER}/{GH_REPO}.git'
subprocess.run(['git', 'clone', '--branch', GH_BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Cloned to', REPO_DIR)

## Cell 3 — Install requirements

In [ ]:
!pip install -q -r requirements.txt

## Cell 4 — Restore data + per-seed baseline LSTMs from Drive

We need from Drive:
1. `data/processed/{train,val,test}.parquet`, `features_all.parquet`, `raw_dataset.parquet`
2. The variant-O baseline LSTM checkpoints `models/lstm_baseline_O_seed{42,43,44}.pt` (and matching scalers) — the baseline LSTM does not depend on the HMM, so we reuse the existing variant-O baselines unchanged for variant Os too.
3. The new variant-Os HMM artifacts `models/hmm_*_Os.joblib` and `data/processed/regime_probabilities_Os.parquet` — these come from the cloned repo (Cell 2 above) since they're committed there.

The cell below copies (1) and (2) from Drive, then verifies that (3) is present in the repo.

In [ ]:
import shutil
from pathlib import Path

src_proc   = Path(PROJECT_DRIVE) / 'data' / 'processed'
src_models = Path(PROJECT_DRIVE) / 'models'
dst_proc   = Path(REPO_DIR) / 'data' / 'processed'
dst_models = Path(REPO_DIR) / 'models'
(dst_proc / 'seeds').mkdir(parents=True, exist_ok=True)
dst_models.mkdir(parents=True, exist_ok=True)

REQUIRED_DATA = [
    'train.parquet', 'val.parquet', 'test.parquet',
    'features_all.parquet', 'raw_dataset.parquet',
]
for fname in REQUIRED_DATA:
    src = src_proc / fname
    if src.exists():
        shutil.copy2(src, dst_proc / fname)
        print(f'  data: {fname}')
    else:
        print(f'  data: {fname}  (MISSING in Drive — abort if not in repo either)')

# variant-O baseline LSTM checkpoints (per seed) and matching scalers — unchanged for Os
for seed in (42, 43, 44):
    for stem in (f'lstm_baseline_O_seed{seed}.pt',
                 f'lstm_baseline_O_seed{seed}_scaler.joblib'):
        src = src_models / stem
        if src.exists():
            shutil.copy2(src, dst_models / stem)
            print(f'  baseline: {stem}')
        else:
            print(f'  baseline: {stem}  (MISSING — needed for ensemble)')

# Verify (3): the new variant-Os HMM artifacts must already be in the cloned repo
required_in_repo = [
    dst_models / 'hmm_winner_Os.joblib',
    dst_models / 'hmm_meta_Os.joblib',
    dst_models / 'hmm_scaler_Os.joblib',
    dst_proc   / 'regime_probabilities_Os.parquet',
]
missing = [f for f in required_in_repo if not f.exists()]
if missing:
    raise FileNotFoundError(
        'Missing variant-Os HMM artifacts in the cloned repo:\n  ' +
        '\n  '.join(str(m) for m in missing) +
        '\nRe-run the local refit script and push the new files to the branch first.'
    )
print('\nAll required variant-Os HMM artifacts present.')

## Cell 5 — Sanity check: verify the variant-Os HMM is the GaussianHMM

Confirms that what we're about to train regime LSTMs against is the clean GaussianHMM partition, not the broken GMMHMM partition. Aborts loudly if the wrong artifacts somehow ended up here.

In [ ]:
import joblib, pandas as pd
m    = joblib.load('models/hmm_winner_Os.joblib')
meta = joblib.load('models/hmm_meta_Os.joblib')
rp   = pd.read_parquet('data/processed/regime_probabilities_Os.parquet')

print(f'HMM class       : {type(m).__name__}')
print(f'n_mix           : {getattr(m, "n_mix", 1)}')
print(f'p_stay diagonal : {m.transmat_.diagonal()}')
print(f'meta            : {meta.get("variant_tag")} — {meta.get("model_class")}')
print()
print(f'p_volatile autocorr lag-1 : {rp["p_volatile"].autocorr(1):.3f}   (expect ~0.97)')
print(f'p_volatile flips above/below 0.5 : {int(((rp["p_volatile"] > 0.5).astype(int).diff() != 0).sum() - 1)}   (expect <100)')

assert type(m).__name__ == 'GaussianHMM', f'Expected GaussianHMM, got {type(m).__name__}'
assert rp['p_volatile'].autocorr(1) > 0.85, 'Posterior autocorrelation too low — wrong HMM artifacts loaded'
print('\n✓ Variant Os HMM is the expected GaussianHMM with clean regime persistence.')

## Cell 6 — Train variant-Os calm + volatile LSTMs (3 seeds)

Calls `src/train_LSTM_regime.py` once per seed (each invocation handles `--regime both`, training calm and volatile sequentially). Output filenames use `_Os_seed{N}` suffix so nothing existing is overwritten.

**LSTM input features for variants O and Os are identical** (`config.LSTM_VARIANT_O_FEATURES`). What changes between O and Os is *which windows* are labelled calm vs volatile (because the HMM posterior changes), and how the ensemble blends them at inference.

In [ ]:
import sys, json, subprocess
from pathlib import Path

sys.path.insert(0, str(Path(REPO_DIR)))
import config

FEATURES = config.LSTM_VARIANT_O_FEATURES   # 5 stationary returns/vol features
SEEDS    = [42, 43, 44]
SCRIPT   = Path(REPO_DIR) / 'src' / 'train_LSTM_regime.py'
HPARAMS_JSON = Path(REPO_DIR) / 'models' / 'hparams_lstm_regime_Os.json'

print(f'LSTM features ({len(FEATURES)}): {FEATURES}')
print(f'Output filename pattern: lstm_{{calm,volatile}}_Os_seed{{42,43,44}}.pt')
print()

def _run_one_seed(args_list, label):
    cmd = [sys.executable, '-u', str(SCRIPT), *args_list]
    print('=' * 80); print(f'[{label}]'); print('Command:', ' '.join(cmd)); print('-' * 80)
    proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end=''); lines.append(line)
    rc = proc.wait()
    return rc, ''.join(lines)

for seed_idx, seed in enumerate(SEEDS):
    label = f'variant Os, seed {seed}'
    suffix = f'_Os_seed{seed}'
    args = [
        '--features', *FEATURES,
        '--regime', 'both',
        '--output-suffix', suffix,
        '--seed', str(seed),
        '--regime-probs-path', 'data/processed/regime_probabilities_Os.parquet',
        '--hmm-meta-path',     'models/hmm_meta_Os.joblib',
    ]
    if seed_idx > 0:
        if HPARAMS_JSON.exists():
            args += ['--fixed-hparams', str(HPARAMS_JSON)]
        else:
            print(f'WARN: {HPARAMS_JSON} missing — seed {seed} will run a full Optuna study')
    rc, text = _run_one_seed(args, label)
    if rc != 0:
        raise RuntimeError(f'{label} failed with return code {rc}')

    # After seed 42: persist its hparams JSON for seeds 43/44 (mirrors nb05's protocol)
    if seed_idx == 0:
        marker = '=== Regime-Specific LSTM Results ==='
        idx = text.find(marker)
        if idx == -1:
            raise RuntimeError(f'Could not find {marker!r} in seed-42 output')
        blob = text[idx + len(marker):].strip()
        results = json.loads(blob)
        HPARAMS_JSON.write_text(json.dumps(results, indent=2))
        print(f'\n[seed 42] hparams JSON saved → {HPARAMS_JSON.name}')

print('\nAll variant Os regime LSTMs trained.')

## Cell 7 — Compute variant-Os ensemble + within-variant DM (inline)

Builds the regime-blended ensemble using the new posteriors and the per-seed (baseline, calm, volatile) predictions. Computes DM(baseline, ensemble) on both losses. Mirrors what `nb06` does for variants O / A / B but for variant Os only and self-contained here.

In [ ]:
import joblib, numpy as np, pandas as pd, torch, scipy.stats as sp_stats
from pathlib import Path
from src.lstm_model import LSTMRegressor, VolatilityWindowDataset
from torch.utils.data import DataLoader
import config

TARGET   = config.LSTM_TARGET
test_df  = pd.read_parquet('data/processed/test.parquet')
rp       = pd.read_parquet('data/processed/regime_probabilities_Os.parquet')

def load_pt(suffix, regime):
    ckpt = torch.load(f'models/lstm_{regime}{suffix}.pt', map_location='cpu', weights_only=False)
    sc = joblib.load(f'models/lstm_{regime}{suffix}_scaler.joblib')
    return ckpt, sc

def _predict(ckpt, scaler, features):
    hp = ckpt['hyperparameters']
    sub = test_df[features + [TARGET]].dropna()
    X = scaler.transform(sub[features].to_numpy(np.float64))
    y = np.log(sub[TARGET].to_numpy(np.float64))
    ds = VolatilityWindowDataset(X, y, hp['seq_len'])
    ldr = DataLoader(ds, batch_size=hp['batch_size'], shuffle=False)
    model = LSTMRegressor(
        input_size=len(features), hidden_size=hp['hidden_size'],
        n_layers=hp['n_layers'], dropout=hp['dropout'],
    )
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in ldr:
            preds.append(model(x).numpy())
    pred = np.exp(np.concatenate(preds))
    idx = sub.index[hp['seq_len'] - 1:]
    return pd.Series(pred, index=idx)

FEATURES = config.LSTM_VARIANT_O_FEATURES
ALL_FEATURES = FEATURES   # both baseline and regime LSTMs use the same input features

seed_dfs = {}
for seed in (42, 43, 44):
    suffix_base = f'_Os_seed{seed}'
    # Baseline LSTM: variant O baseline is unchanged for Os (HMM-independent)
    base_ck, base_sc = load_pt(f'_O_seed{seed}', 'baseline')
    pred_base = _predict(base_ck, base_sc, ALL_FEATURES)

    # Regime LSTMs: trained in Cell 6 with _Os_seedN suffix
    calm_ck, calm_sc = load_pt(suffix_base, 'calm')
    vol_ck,  vol_sc  = load_pt(suffix_base, 'volatile')
    pred_calm = _predict(calm_ck, calm_sc, ALL_FEATURES)
    pred_vol  = _predict(vol_ck,  vol_sc,  ALL_FEATURES)

    df = pd.concat([pred_base.rename('baseline'),
                    pred_calm.rename('calm'),
                    pred_vol.rename('volatile')], axis=1)
    df['target'] = test_df[TARGET]
    df = df.join(rp[['p_calm', 'p_volatile']], how='inner').dropna()
    df['ensemble'] = df['p_calm'] * df['calm'] + df['p_volatile'] * df['volatile']
    out = Path('data/processed/seeds') / f'test_predictions_Os_seed{seed}.parquet'
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out)
    print(f'Wrote {out}  ({len(df)} rows)')
    seed_dfs[seed] = df

# Mean-of-seeds ensemble
common_idx = seed_dfs[42].index
for s in (43, 44):
    common_idx = common_idx.intersection(seed_dfs[s].index)

mean_df = pd.DataFrame(index=common_idx)
mean_df['target']   = seed_dfs[42].loc[common_idx, 'target']
mean_df['baseline'] = sum(seed_dfs[s].loc[common_idx, 'baseline'] for s in (42, 43, 44)) / 3
mean_df['ensemble'] = sum(seed_dfs[s].loc[common_idx, 'ensemble'] for s in (42, 43, 44)) / 3
mean_df['calm']     = sum(seed_dfs[s].loc[common_idx, 'calm']     for s in (42, 43, 44)) / 3
mean_df['volatile'] = sum(seed_dfs[s].loc[common_idx, 'volatile'] for s in (42, 43, 44)) / 3
mean_df = mean_df.join(rp[['p_calm', 'p_volatile']], how='inner').dropna()
out = Path('data/processed/test_predictions_Os.parquet')
mean_df.to_parquet(out)
print(f'\nWrote mean-of-3-seeds ensemble: {out}  ({len(mean_df)} rows)')

# Within-variant DM(baseline, ensemble) — same function as nb06
def diebold_mariano(y_true, y_a, y_b, h=21, loss='mse'):
    y_true, y_a, y_b = (np.asarray(x, float) for x in (y_true, y_a, y_b))
    e_a = (y_a - y_true)**2 if loss == 'mse' else np.abs(y_a - y_true)
    e_b = (y_b - y_true)**2 if loss == 'mse' else np.abs(y_b - y_true)
    d = e_a - e_b
    n = len(d); d_bar = float(d.mean())
    L = max(h - 1, 0); S = float(np.var(d, ddof=0))
    for k in range(1, L + 1):
        w = 1.0 - k / (L + 1)
        S += 2.0 * w * float(np.mean((d[k:] - d_bar) * (d[:-k] - d_bar)))
    if S <= 0: S = float(np.var(d, ddof=0))
    dm_raw = d_bar / np.sqrt(S / n)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm = float(dm_raw * hln)
    p = float(2.0 * (1.0 - sp_stats.t.cdf(np.abs(dm), df=n - 1)))
    return dm, p

y_true = mean_df['target'].values
bl     = mean_df['baseline'].values
en     = mean_df['ensemble'].values
dm_mse, p_mse = diebold_mariano(y_true, bl, en, h=21, loss='mse')
dm_mae, p_mae = diebold_mariano(y_true, bl, en, h=21, loss='mae')

print('\n' + '=' * 70)
print('Variant Os — within-variant DM(baseline, ensemble) on mean-of-seeds')
print('=' * 70)
print(f'  n = {len(mean_df)}')
print(f'  DM_MSE = {dm_mse:+.3f}   p_MSE = {p_mse:.4f}   '
      f"verdict: {'ensemble wins' if p_mse < 0.05 and dm_mse > 0 else ('baseline wins' if p_mse < 0.05 else 'tie')}")
print(f'  DM_MAE = {dm_mae:+.3f}   p_MAE = {p_mae:.4f}   '
      f"verdict: {'ensemble wins' if p_mae < 0.05 and dm_mae > 0 else ('baseline wins' if p_mae < 0.05 else 'tie')}")

# Quick metrics summary
from src.utils import regression_metrics
print('\nMean-of-seeds test metrics (Os ensemble vs target):')
for k, v in regression_metrics(y_true, en).items():
    print(f'  {k:>5}: {v:.6f}')

## Cell 8 — Sync new variant-Os artifacts back to Drive

In [ ]:
import shutil, glob
from pathlib import Path

drive_models = Path(PROJECT_DRIVE) / 'models'
drive_proc   = Path(PROJECT_DRIVE) / 'data' / 'processed'
drive_seeds  = drive_proc / 'seeds'
for p in (drive_models, drive_proc, drive_seeds):
    p.mkdir(parents=True, exist_ok=True)

# Push only the new Os-specific artifacts; nothing else is touched
patterns = [
    ('models', 'lstm_calm_Os_seed*.pt'),
    ('models', 'lstm_calm_Os_seed*_scaler.joblib'),
    ('models', 'lstm_volatile_Os_seed*.pt'),
    ('models', 'lstm_volatile_Os_seed*_scaler.joblib'),
    ('models', 'hmm_*_Os.joblib'),
    ('models', 'hparams_lstm_regime_Os.json'),
    ('data/processed', 'regime_probabilities_Os.parquet'),
    ('data/processed', 'test_predictions_Os.parquet'),
    ('data/processed/seeds', 'test_predictions_Os_seed*.parquet'),
]
for src_root, pattern in patterns:
    for f in glob.glob(f'{src_root}/{pattern}'):
        dst = (drive_models if src_root == 'models'
               else drive_seeds if src_root.endswith('seeds')
               else drive_proc) / Path(f).name
        shutil.copy2(f, dst)
        print(f'  → {dst}')
print('\nDone. Pull the listed files into your local repo to regenerate paper figures and tables.')